In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/anishapanja/brca-pancancer-atlas-clinical/data_clinical_patient.txt
/kaggle/input/datasets/anishapanja/bc-xai-macenko-reference-a2e2/reference_tile.png


In [2]:
import os
import pandas as pd

INPUT_ROOT = "/kaggle/input"

for root, dirs, files in os.walk(INPUT_ROOT):
    for file in files:
        if file == "data_clinical_patient.txt":
            clinical_path = os.path.join(root, file)

print("Clinical file path:", clinical_path)

df = pd.read_csv(clinical_path, sep="\t", comment="#", low_memory=False)

print("Shape:", df.shape)
print("Columns:")
print(df.columns.tolist())

print("\nRequired columns present:")
print("PATIENT_ID:", "PATIENT_ID" in df.columns)
print("SUBTYPE:", "SUBTYPE" in df.columns)

c8 = df[
    df["PATIENT_ID"].astype(str).str.startswith("TCGA-C8-")
    & df["SUBTYPE"].notna()
    & (df["SUBTYPE"] != "BRCA_Normal")
]

print("\nC8 labeled patients:", c8["PATIENT_ID"].nunique())
print(c8["SUBTYPE"].value_counts())
print(c8[["PATIENT_ID", "SUBTYPE"]].head(10))

Clinical file path: /kaggle/input/datasets/anishapanja/brca-pancancer-atlas-clinical/data_clinical_patient.txt
Shape: (1084, 38)
Columns:
['PATIENT_ID', 'SUBTYPE', 'CANCER_TYPE_ACRONYM', 'OTHER_PATIENT_ID', 'AGE', 'SEX', 'AJCC_PATHOLOGIC_TUMOR_STAGE', 'AJCC_STAGING_EDITION', 'DAYS_LAST_FOLLOWUP', 'DAYS_TO_BIRTH', 'DAYS_TO_INITIAL_PATHOLOGIC_DIAGNOSIS', 'ETHNICITY', 'FORM_COMPLETION_DATE', 'HISTORY_NEOADJUVANT_TRTYN', 'ICD_10', 'ICD_O_3_HISTOLOGY', 'ICD_O_3_SITE', 'INFORMED_CONSENT_VERIFIED', 'NEW_TUMOR_EVENT_AFTER_INITIAL_TREATMENT', 'PATH_M_STAGE', 'PATH_N_STAGE', 'PATH_T_STAGE', 'PERSON_NEOPLASM_CANCER_STATUS', 'PRIMARY_LYMPH_NODE_PRESENTATION_ASSESSMENT', 'PRIOR_DX', 'RACE', 'RADIATION_THERAPY', 'WEIGHT', 'IN_PANCANPATHWAYS_FREEZE', 'OS_STATUS', 'OS_MONTHS', 'DSS_STATUS', 'DSS_MONTHS', 'DFS_STATUS', 'DFS_MONTHS', 'PFS_STATUS', 'PFS_MONTHS', 'GENETIC_ANCESTRY_LABEL']

Required columns present:
PATIENT_ID: True
SUBTYPE: True

C8 labeled patients: 47
SUBTYPE
BRCA_Her2     15
BRCA_LumB 

In [3]:
# Install OpenSlide system tools for reading SVS whole-slide images.
import subprocess
subprocess.run(["apt-get", "install", "-y", "openslide-tools", "-q"], check=True)

# Install only libraries that work on Kaggle Python 3.12.
# Do NOT install staintools because it imports missing spams on Python 3.12.
subprocess.run(["pip", "install", "openslide-python", "scikit-image", "requests", "-q"], check=True)

# Import standard libraries.
import os
import gc
import csv
import json
import shutil
import random
import traceback
from pathlib import Path

# Import scientific and image-processing libraries.
import cv2
import numpy as np
import pandas as pd
import requests
import openslide
from PIL import Image

# Fixed random seed for reproducibility.
SEED = 42
rng = random.Random(SEED)

# Site and batch settings.
SITE_CODE = "C8"
BATCH_SIZE = 15

# Frozen preprocessing parameters.
PATCH_SIZE = 256
STRIDE = 256
TARGET_MPP = 1.0
MIN_TISSUE_FRACTION = 0.50
SATURATION_THRESHOLD = 15
MAX_PATCHES_PER_PATIENT = 2000
JPEG_QUALITY = 95

# Output paths.
WORK_DIR = Path("/kaggle/working")
PATCH_ROOT = WORK_DIR / "patches" / SITE_CODE
TMP_DIR = WORK_DIR / "tmp_svs"
LOG_PATH = WORK_DIR / "preprocessing_log.csv"

# Create output folders.
PATCH_ROOT.mkdir(parents=True, exist_ok=True)
TMP_DIR.mkdir(parents=True, exist_ok=True)

# Find an input file by exact filename.
def find_input_file(filename):
    for root, dirs, files in os.walk("/kaggle/input"):
        if filename in files:
            return Path(root) / filename
    raise FileNotFoundError(f"Could not find {filename} under /kaggle/input")

# Locate clinical labels and reference tile.
clinical_path = find_input_file("data_clinical_patient.txt")
reference_path = find_input_file("reference_tile.png")

# Load clinical data.
clinical_df = pd.read_csv(clinical_path, sep="\t", comment="#", low_memory=False)

# Keep labeled C8 patients only.
c8_df = clinical_df[
    clinical_df["PATIENT_ID"].astype(str).str.startswith("TCGA-C8-")
    & clinical_df["SUBTYPE"].notna()
    & (clinical_df["SUBTYPE"] != "BRCA_Normal")
].copy()

# Sort for deterministic batch order.
c8_df = c8_df.sort_values("PATIENT_ID").reset_index(drop=True)

# Select first 15 patients for batch 1.
batch_df = c8_df.head(BATCH_SIZE).copy()

# Print sanity checks.
print("Clinical file:", clinical_path)
print("Reference tile:", reference_path)
print("Total labeled C8 patients:", c8_df["PATIENT_ID"].nunique())
print("Batch 1 patients:", batch_df["PATIENT_ID"].tolist())
print("Batch 1 subtype counts:")
print(batch_df["SUBTYPE"].value_counts())

# Luminosity standardization similar to staintools.
def standardize_luminosity(img):
    img = img.astype(np.float32)
    p = np.percentile(img, 95)
    if p <= 0:
        return np.clip(img, 0, 255).astype(np.uint8)
    img = img * (255.0 / p)
    return np.clip(img, 0, 255).astype(np.uint8)

# Convert RGB to optical density.
def rgb_to_od(img):
    img = img.astype(np.float32)
    return -np.log((img + 1.0) / 255.0)

# Convert optical density back to RGB.
def od_to_rgb(od):
    rgb = 255.0 * np.exp(-od)
    return np.clip(rgb, 0, 255).astype(np.uint8)

# Estimate Macenko H&E stain matrix.
def get_macenko_stain_matrix(img, beta=0.15, alpha=1):
    img = standardize_luminosity(img)
    od = rgb_to_od(img).reshape((-1, 3))
    od = od[np.all(od > beta, axis=1)]

    if od.shape[0] < 100:
        raise RuntimeError("Too few tissue pixels for Macenko fitting.")

    _, _, vh = np.linalg.svd(od, full_matrices=False)
    top_vectors = vh[:2].T

    projected = np.dot(od, top_vectors)
    angles = np.arctan2(projected[:, 1], projected[:, 0])

    min_angle = np.percentile(angles, alpha)
    max_angle = np.percentile(angles, 100 - alpha)

    v1 = np.dot(top_vectors, np.array([np.cos(min_angle), np.sin(min_angle)]))
    v2 = np.dot(top_vectors, np.array([np.cos(max_angle), np.sin(max_angle)]))

    if v1[0] > v2[0]:
        he = np.stack([v1, v2], axis=1)
    else:
        he = np.stack([v2, v1], axis=1)

    he = he / np.linalg.norm(he, axis=0, keepdims=True)
    return he

# Estimate stain concentrations.
def get_concentrations(img, stain_matrix):
    od = rgb_to_od(img).reshape((-1, 3)).T
    concentrations, _, _, _ = np.linalg.lstsq(stain_matrix, od, rcond=None)
    return concentrations

# Fit fixed reference Macenko target once.
reference_img = np.array(Image.open(reference_path).convert("RGB"))
reference_img = standardize_luminosity(reference_img)
reference_he = get_macenko_stain_matrix(reference_img)
reference_conc = get_concentrations(reference_img, reference_he)
reference_max = np.percentile(reference_conc, 99, axis=1)

# Normalize one patch to the fixed reference tile.
def normalize_patch(rgb_patch):
    try:
        patch = standardize_luminosity(rgb_patch)
        source_he = get_macenko_stain_matrix(patch)
        source_conc = get_concentrations(patch, source_he)
        source_max = np.percentile(source_conc, 99, axis=1)
        scale = reference_max / (source_max + 1e-8)
        source_conc = source_conc * scale[:, None]
        normalized_od = np.dot(reference_he, source_conc).T.reshape(patch.shape)
        normalized_rgb = od_to_rgb(normalized_od)
        return normalized_rgb
    except Exception:
        return rgb_patch

# Append one row to the preprocessing log.
def write_log(row):
    file_exists = LOG_PATH.exists()
    with open(LOG_PATH, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=[
                "patient_id",
                "subtype",
                "status",
                "patches_saved",
                "tissue_patches_seen",
                "svs_files_found",
                "error_message",
            ],
        )
        if not file_exists:
            writer.writeheader()
        writer.writerow(row)

# Free disk space in GB.
def free_disk_gb(path="/kaggle/working"):
    usage = shutil.disk_usage(path)
    return usage.free / (1024 ** 3)

# Query GDC for open SVS slide files for one patient.
def query_gdc_svs_files(patient_id):
    filters = {
        "op": "and",
        "content": [
            {"op": "in", "content": {"field": "cases.submitter_id", "value": [patient_id]}},
            {"op": "in", "content": {"field": "data_type", "value": ["Slide Image"]}},
            {"op": "in", "content": {"field": "access", "value": ["open"]}},
        ],
    }

    params = {
        "filters": json.dumps(filters),
        "fields": "file_id,file_name,file_size,cases.submitter_id",
        "format": "JSON",
        "size": "200",
    }

    response = requests.get("https://api.gdc.cancer.gov/files", params=params, timeout=60)
    response.raise_for_status()
    hits = response.json()["data"]["hits"]

    svs_hits = [h for h in hits if str(h.get("file_name", "")).lower().endswith(".svs")]
    svs_hits = sorted(svs_hits, key=lambda x: x.get("file_name", ""))
    return svs_hits

# Stream-download one SVS file.
def download_svs(file_id, out_path):
    url = f"https://api.gdc.cancer.gov/data/{file_id}"

    with requests.get(url, stream=True, timeout=120) as response:
        response.raise_for_status()
        with open(out_path, "wb") as f:
            for chunk in response.iter_content(chunk_size=8192):
                if chunk:
                    f.write(chunk)

    if out_path.stat().st_size <= 1_000_000:
        raise RuntimeError(f"Downloaded SVS is too small: {out_path.stat().st_size} bytes")

# Choose OpenSlide level closest to 1 micrometer per pixel.
def choose_level_for_target_mpp(slide):
    mpp_x = slide.properties.get("openslide.mpp-x")

    if mpp_x is not None:
        base_mpp = float(mpp_x)
        target_downsample = TARGET_MPP / base_mpp
    else:
        target_downsample = 4.0

    downsamples = [float(d) for d in slide.level_downsamples]
    level = int(np.argmin([abs(d - target_downsample) for d in downsamples]))
    return level

# Tissue detection using HSV saturation.
def tissue_fraction_hsv(rgb_patch):
    hsv = cv2.cvtColor(rgb_patch, cv2.COLOR_RGB2HSV)
    saturation = hsv[:, :, 1]
    return float(np.mean(saturation > SATURATION_THRESHOLD))

# Save RGB patch as JPEG.
def save_jpeg(rgb_patch, out_path):
    Image.fromarray(rgb_patch).save(out_path, format="JPEG", quality=JPEG_QUALITY)

# Process one SVS slide into patient patch folder.
def process_slide_into_patient_reservoir(slide_path, patient_dir, patient_id, current_saved, tissue_seen):
    slide = openslide.open_slide(str(slide_path))
    level = choose_level_for_target_mpp(slide)
    level_w, level_h = slide.level_dimensions[level]
    downsample = float(slide.level_downsamples[level])

    print("Selected OpenSlide level:", level)
    print("Level dimensions:", level_w, level_h)
    print("Level downsample:", downsample)

    for y in range(0, level_h - PATCH_SIZE + 1, STRIDE):
        for x in range(0, level_w - PATCH_SIZE + 1, STRIDE):
            level0_x = int(x * downsample)
            level0_y = int(y * downsample)

            patch_rgba = slide.read_region((level0_x, level0_y), level, (PATCH_SIZE, PATCH_SIZE))
            patch_rgb = np.array(patch_rgba.convert("RGB"))

            if tissue_fraction_hsv(patch_rgb) < MIN_TISSUE_FRACTION:
                continue

            tissue_seen += 1
            patch_rgb = normalize_patch(patch_rgb)

            if current_saved < MAX_PATCHES_PER_PATIENT:
                out_index = current_saved
                current_saved += 1
            else:
                replacement_index = rng.randint(0, tissue_seen - 1)
                if replacement_index >= MAX_PATCHES_PER_PATIENT:
                    continue
                out_index = replacement_index

            out_name = f"{patient_id}_{out_index:05d}.jpg"
            out_path = patient_dir / out_name
            save_jpeg(patch_rgb, out_path)

    slide.close()
    return current_saved, tissue_seen

# Process each patient in batch 1.
for row_index, row in batch_df.iterrows():
    patient_id = row["PATIENT_ID"]
    subtype = row["SUBTYPE"]
    patient_dir = PATCH_ROOT / patient_id

    print("\n" + "=" * 80)
    print(f"Patient {row_index + 1}/{len(batch_df)}: {patient_id} | {subtype}")
    print(f"Free disk before patient: {free_disk_gb():.2f} GB")

    if patient_dir.exists() and any(patient_dir.glob("*.jpg")):
        existing_count = len(list(patient_dir.glob("*.jpg")))
        print(f"Skipping existing patient folder with {existing_count} JPEGs.")
        write_log({
            "patient_id": patient_id,
            "subtype": subtype,
            "status": "skipped_existing",
            "patches_saved": existing_count,
            "tissue_patches_seen": "",
            "svs_files_found": "",
            "error_message": "",
        })
        continue

    if free_disk_gb() < 2.0:
        print("Stopping safely because free disk is below 2 GB.")
        break

    patient_dir.mkdir(parents=True, exist_ok=True)

    current_saved = 0
    tissue_seen = 0
    svs_files = []

    try:
        svs_files = query_gdc_svs_files(patient_id)
        print(f"SVS files found: {len(svs_files)}")

        if len(svs_files) == 0:
            raise RuntimeError("No open SVS files found in GDC.")

        for svs_index, svs in enumerate(svs_files):
            if free_disk_gb() < 2.0:
                raise RuntimeError("Free disk dropped below 2 GB before download.")

            file_id = svs["file_id"]
            file_name = svs["file_name"]
            svs_path = TMP_DIR / f"{patient_id}_{svs_index}_{file_name}"

            print(f"Downloading SVS {svs_index + 1}/{len(svs_files)}: {file_name}")
            download_svs(file_id, svs_path)

            print(f"Processing slide: {svs_path.name}")
            current_saved, tissue_seen = process_slide_into_patient_reservoir(
                svs_path,
                patient_dir,
                patient_id,
                current_saved,
                tissue_seen,
            )

            if svs_path.exists():
                os.remove(svs_path)

            gc.collect()

            print(f"Patches saved so far: {current_saved}")
            print(f"Tissue patches seen so far: {tissue_seen}")
            print(f"Free disk after slide: {free_disk_gb():.2f} GB")

        status = "success" if current_saved > 0 else "failed_no_patches"

        write_log({
            "patient_id": patient_id,
            "subtype": subtype,
            "status": status,
            "patches_saved": current_saved,
            "tissue_patches_seen": tissue_seen,
            "svs_files_found": len(svs_files),
            "error_message": "",
        })

        print(f"Finished {patient_id}: {status}, saved {current_saved} patches.")

    except Exception as e:
        error_message = repr(e)
        print(f"FAILED {patient_id}: {error_message}")
        print(traceback.format_exc())

        for leftover in TMP_DIR.glob(f"{patient_id}_*"):
            try:
                os.remove(leftover)
            except Exception:
                pass

        write_log({
            "patient_id": patient_id,
            "subtype": subtype,
            "status": "failed",
            "patches_saved": current_saved,
            "tissue_patches_seen": tissue_seen,
            "svs_files_found": len(svs_files),
            "error_message": error_message,
        })

    print(f"Free disk after patient: {free_disk_gb():.2f} GB")

# Final summary.
print("\n" + "=" * 80)
print("C8 batch 1 preprocessing finished.")
print("Patch output folder:", PATCH_ROOT)
print("Log file:", LOG_PATH)

summary_rows = []
for patient_folder in sorted(PATCH_ROOT.glob("TCGA-C8-*")):
    summary_rows.append({
        "patient_id": patient_folder.name,
        "jpg_count": len(list(patient_folder.glob("*.jpg"))),
    })

summary_df = pd.DataFrame(summary_rows)
print(summary_df)

if LOG_PATH.exists():
    print(pd.read_csv(LOG_PATH))

Reading package lists...
Building dependency tree...
Reading state information...
The following additional packages will be installed:
  libopenslide0
Suggested packages:
  libtiff-tools
The following NEW packages will be installed:
  libopenslide0 openslide-tools
0 upgraded, 2 newly installed, 0 to remove and 83 not upgraded.
Need to get 104 kB of archives.
After this operation, 297 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libopenslide0 amd64 3.4.1+dfsg-5build1 [89.8 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/universe amd64 openslide-tools amd64 3.4.1+dfsg-5build1 [13.8 kB]
Fetched 104 kB in 0s (1,272 kB/s)
Selecting previously unselected package libopenslide0.
(Reading database ... 121026 files and directories currently installed.)
Preparing to unpack .../libopenslide0_3.4.1+dfsg-5build1_amd64.deb ...
Unpacking libopenslide0 (3.4.1+dfsg-5build1) ...
Selecting previously unselected package openslide-tools.
Preparing to

/tmp/ipykernel_16/2153965174.py:105: RuntimeWarning: overflow encountered in exp
  rgb = 255.0 * np.exp(-od)
/tmp/ipykernel_16/2153965174.py:105: RuntimeWarning: overflow encountered in multiply
  rgb = 255.0 * np.exp(-od)


Patches saved so far: 501
Tissue patches seen so far: 501
Free disk after slide: 19.47 GB
Processing slide: TCGA-C8-A12K_1_TCGA-C8-A12K-01A-02-TSB.6423d4b2-7584-4f9b-8525-9429aebc8f0f.svs
Selected OpenSlide level: 1
Level dimensions: 26403 8125
Level downsample: 4.000203552626595
Patches saved so far: 1034
Tissue patches seen so far: 1034
Free disk after slide: 19.44 GB
Processing slide: TCGA-C8-A12K_2_TCGA-C8-A12K-01Z-00-DX1.D71C7974-65F7-4EF0-8FFB-9F27CEC85242.svs
Selected OpenSlide level: 1
Level dimensions: 18801 12508
Level downsample: 4.000199706239387
Patches saved so far: 2000
Tissue patches seen so far: 3508
Free disk after slide: 19.38 GB
Finished TCGA-C8-A12K: success, saved 2000 patches.
Free disk after patient: 19.38 GB

Patient 2/15: TCGA-C8-A12L | BRCA_Her2
Free disk before patient: 19.38 GB
SVS files found: 3
Processing slide: TCGA-C8-A12L_0_TCGA-C8-A12L-01A-01-BSA.f97cb175-3951-4f67-ac52-18ad07e74726.svs
Selected OpenSlide level: 1
Level dimensions: 23522 7690
Level do